<!-- nav -->
[← 6 · Analyzing and rewriting constraints](06_Analyzing_And_Rewriting_Constraints.ipynb) · [Home](../../README.md) · [8 · Constraints for firmware and hardware →](08_Constraints_For_Firmware_And_Hardware.ipynb)

# 7 · One constraint, many languages

**The problem.** Back to the four copies of case study 1. The QA team checks shipments in Excel; the data pipeline is
Python; the analysts use MATLAB; the specification is a LaTeX document. Each wants the constraint in its own language,
and each copy, written by hand, drifts.

mbse-expressions models each of those languages as a **dialect**: its expressions are data too, it has its own
evaluator, and it writes its own source text. **Translators** between dialects turn the neutral constraint (in the Basic
dialect, which the earlier case studies used) into each, and back.

In [1]:
import { Evaluators, Expressions as E } from "@mbse/expressions";
import * as Excel from "@mbse/expressions/Dialects/Excel";
import * as Latex from "@mbse/expressions/Dialects/Latex";
import * as Matlab from "@mbse/expressions/Dialects/Matlab";
import * as Python from "@mbse/expressions/Dialects/Python";
import { Terms } from "@mbse/expressions/Framework";
import * as Translators from "@mbse/expressions/Translators";
import { cold_chain, show } from "./toolkit.ts";

const store = cold_chain();
const [self, c] = [E.variable("this"), E.variable("c")];
const in_range = E.literal(2.0).le(self.celsius).and_(self.celsius.le(8.0)).data;
const allowed = self.readings.count_where("c", c.gt(8.0)).le(2n).and_(self.readings.max().le(12.0)).data;
const languages: [any, any][] = [[Excel.Expressions.DIALECT, Excel.Text], [Python.Expressions.DIALECT, Python.Text],
  [Matlab.Expressions.DIALECT, Matlab.Text], [Latex.Expressions.DIALECT, Latex.Text]];

## Step 1: translating

`Translators.between(source, target)` finds the translator, and `forward` translates. Each dialect's `Text.ToText`
writes its source:

In [2]:
for (const [dialect, text] of languages) {
  console.log(dialect.name().padEnd(7), text.ToText(Translators.between(E.DIALECT, dialect).forward(in_range)));
}

Excel   =AND(2.0 <= this.celsius, this.celsius <= 8.0)


Python  2.0 <= this.celsius and this.celsius <= 8.0


Matlab  2.0 <= this.celsius && this.celsius <= 8.0


Latex   2.0 \leq \mathit{this}.\mathit{celsius} \land \mathit{this}.\mathit{celsius} \leq 8.0


Collections translate to each language's idiom: `MAP` and `LAMBDA` in Excel, generator expressions in Python,
`arrayfun` in MATLAB. LaTeX has no counterpart for a quantifier, and says so:

In [3]:
for (const [dialect, text] of languages) {
  try {
    console.log(dialect.name().padEnd(7), text.ToText(Translators.between(E.DIALECT, dialect).forward(allowed)));
  } catch (error) {
    console.log(dialect.name().padEnd(7), (error as Error).message);
  }
}

Excel   =AND(SUM(MAP(this.readings, LAMBDA(c, IF(c > 8.0, 1, 0)))) <= 2, MAX(this.readings) <= 12.0)


Python  sum(1 for c in this.readings if c > 8.0) <= 2 and max(this.readings) <= 12.0


Matlab  nnz(arrayfun(@(c) c > 8.0, this.readings)) <= 2 && max(this.readings) <= 12.0


Latex   Basic quantifier 'count' has no Latex counterpart


## Step 2: each language evaluates by its own rules

A translation keeps the constraint's **form**. Its value is the target language's business, because that's where it will
run. Evaluate the translations on a shipment whose logger never reported:

In [4]:
const silent = store.Shipment().id("S3").create();
const excel = Translators.between(E.DIALECT, Excel.Expressions.DIALECT).forward(in_range);
const python = Translators.between(E.DIALECT, Python.Expressions.DIALECT).forward(in_range);
console.log("Basic  ", Evaluators.OfAny(in_range, { this: silent }));
console.log("Excel  ", String(Excel.Evaluators.OfAny(excel, { this: silent })));
try {
  Python.Evaluators.OfAny(python, { this: silent });
} catch (error) {
  console.log("Python ", String(error));
}

Basic   null


Excel   #FIELD!


Python  AttributeError: 'Shipment' object has no attribute 'celsius'


Basic says unknown; Excel gives the error value `#FIELD!`, which propagates through a worksheet the way errors do
there; Python raises, as `shipment.celsius` would. Coercion differs too. Excel compares an integer limit with a float
reading happily, where Basic says unknown (case study 2):

In [5]:
const sloppy = self.celsius.ge(2n).data;
const cold = store.Shipment().id("S1").celsius(5.5).create();
console.log("Basic  ", Evaluators.OfAny(sloppy, { this: cold }));
console.log("Excel  ", Excel.Evaluators.OfAny(Translators.between(E.DIALECT, Excel.Expressions.DIALECT).forward(sloppy), { this: cold }));

Basic   null


Excel   true


So keep the constraint in Basic, where its meaning is fixed, translate it to where it is checked, and test it there with
that language's evaluator, which models its rules exactly.

## Step 3: reading constraints back

A constraint written in another language can be read into Basic, to analyze it (case study 6) or store it. Python's
dialect builds its expressions as Python's `ast` module has them (`compare`, `attribute`, `boolop` and so on), and the
translator goes the other way. (In Python, `Text.FromText` also parses source.)

In [6]:
const P = Python.Expressions;
const celsius = P.attribute(P.name("this"), "celsius");
const parsed = P.boolop("and", P.compare("<=", 2.0, celsius), P.compare("<=", celsius, 8.0));
console.log(Python.Text.ToText(parsed));
const back = Translators.between(Python.Expressions.DIALECT, E.DIALECT).forward(parsed);
console.log(show(back as any));
console.log(Terms.same(back, in_range));

2.0 <= this.celsius and this.celsius <= 8.0


and(le(2.0, get(this, 'celsius')), le(get(this, 'celsius'), 8.0))


true


Round trips keep what both languages can say. What one of them can't, such as a let in MATLAB, which has none, is
written another way, and comes back in that other form.

## Step 4: what a constraint may reach

A Python or MATLAB expression can call functions, and an expression loaded from a file mustn't be able to call
anything it likes. Each dialect's scope reaches only what the caller gives it: here, a model of Python's `math`, and
nothing else:

In [7]:
const rounded = P.import_("math", P.compare("<=", P.call(P.attribute(P.name("math"), "floor"), celsius), 8n));
console.log(Python.Text.ToText(rounded));
try {
  Python.Evaluators.OfAny(rounded, { this: cold });
} catch (error) {
  console.log(String(error));
}
const math = new Python.Evaluators.Module("math", { floor: (x: number) => BigInt(Math.floor(x)) });
console.log(Python.Evaluators.OfAny(rounded, new Python.Evaluators.Scope({ this: cold }, { modules: { math } })));

import math
math.floor(this.celsius) <= 8


ImportError: import of 'math' is not allowed by the scope


true


## Why it works this way

- **One translator per pair, by rules.** Each pair of dialects has its own translator, declared as rules pairing
  patterns, as the rewriting of case study 6 was. Translating directly or through Basic gives the same result.
- **Form, not value.** A translator can't make Excel treat a missing value as unknown, and shouldn't try: the
  translated constraint must read the way an engineer in that language would write it.
- **Nothing is reachable by default.** Constraints travel as data, sometimes from people you don't fully trust. A scope
  is an allowlist.

<!-- nav -->
[← 6 · Analyzing and rewriting constraints](06_Analyzing_And_Rewriting_Constraints.ipynb) · [Home](../../README.md) · [8 · Constraints for firmware and hardware →](08_Constraints_For_Firmware_And_Hardware.ipynb)